# Demo de Structured Streaming (Socket → WordCount) no VS Code (Windows)

## Visão geral do projeto
Este projeto demonstra um pipeline simples de **Structured Streaming com Apache Spark** usando uma fonte via **socket TCP** (porta `9999`).  
O fluxo de texto (linhas digitadas no terminal) é transformado em palavras e o Spark mantém uma **contagem acumulada** (running word count).

Como estamos usando **notebooks no VS Code (Windows)**, o sink `console` pode não aparecer no output da célula. Por isso, usamos **Memory Sink** (`format("memory")`) e consultamos o resultado como uma “tabela temporária” via SQL com `queryName("wc")`.


## Pré-requisitos
- Python + PySpark (ou ambiente Spark com PySpark funcional).
- VS Code com extensão de Jupyter/Notebooks.
- Utilitário de socket TCP no Windows:
  - Recomendado: `nc` se você tiver instalado com WSL ubuntu
  - Alternativa: [Instalar o emulador de linux ](https://mobaxterm.mobatek.net/download-home-edition.html)
- Com wsl ou mobaxterm no termina digite: `nc -lk 9999` para escutar a porta 9999

# Inicialização do Spark (SparkSession)

Objetivo: criar (ou reutilizar) a SparkSession, que é o ponto de entrada para usar DataFrames e Structured Streaming.

O que este notebook faz:

Importa SparkSession.

Cria a sessão Spark com um nome de aplicação.

Deixa a variável spark disponível para os próximos notebooks.

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode
from pyspark.sql.functions import split

spark = SparkSession \
    .builder \
    .appName("StructuredNetworkWordCount") \
    .getOrCreate()

# Definição do Stream e Transformações (word count)

Objetivo: definir a leitura contínua do socket (TCP) e preparar as transformações para gerar a contagem de palavras.

Pré-requisito:

O servidor precisa estar rodando e “escutando” na porta 9999, por exemplo:

Linux/WSL: nc -lk 9999

O que este notebook faz:

Cria um DataFrame de streaming (lines) lendo texto do socket.

Transforma cada linha em várias linhas (cada uma com uma palavra).

Agrupa e conta as palavras (resultado contínuo em wordCounts).

In [10]:
# DataFrame de streaming: cada linha do socket vira um registro em `value`
lines = (
    spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)

# Quebra linhas em palavras:
# - split: divide a string por espaço
# - explode: “explode” a lista em várias linhas
words = lines.select(
    explode(
        split(lines.value, " ")
    ).alias("word")
)

# Contagem contínua de palavras
wordCounts = words.groupBy("word").count()

# Execução do Streaming + Visualização + Encerramento
Objetivo: iniciar a query de streaming, armazenar o resultado em memória e ficar consultando periodicamente para visualizar a contagem atualizada. Ao final (ou ao interromper), parar o pipeline de forma limpa.

Por que usamos format("memory")?

Em notebook, o sink console frequentemente não aparece no output da célula. O memory cria uma “tabela temporária” (por queryName) que você consegue consultar com SQL e ver na saída do notebook.

O que este notebook faz:

Inicia o stream (background) escrevendo em memória.

Consulta a tabela wc a cada 2 segundos para exibir a contagem.

Se você interromper (Stop/Interrupt), ele executa o finally para parar a query.

In [11]:
import time

# Inicia a query de streaming em background e materializa em memória como uma tabela chamada "wc"
query = (
    wordCounts.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("wc")
    .start()
)

try:
    # Loop de visualização: a cada 2s mostra a contagem atual
    for _ in range(1000):
        spark.sql("select * from wc order by count desc").show(truncate=False)
        time.sleep(2)

except KeyboardInterrupt:
    # Se você interromper manualmente, cai aqui (nem sempre aparece no VS Code, mas é ok)
    pass

finally:
    # Encerra a query de forma limpa (para a “escuta” do socket)
    if query.isActive:
        query.stop()
        query.awaitTermination(10)  # opcional: espera encerrar totalmente



+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-----+

+----+-----+
|word|count|
+----+-----+
+----+-

# Como parar manualmente (sem esperar o loop acabar):

Você pode clicar em Interrupt/Stop no notebook; o finally vai executar e parar a query.

Ou pode criar uma célula separada com:

In [ ]:
if query.isActive:
    query.stop()

Como confirmar se ainda há streams ativos:

In [12]:
spark.streams.active

[]

Como parar tudo que estiver ativo (caso você tenha mais de um stream):

In [13]:
for q in spark.streams.active:
    q.stop()

### Checklist rápido de funcionamento

Terminal com nc/ncat rodando e você digitando linhas + ENTER ✅

Notebook 1 executado (SparkSession criada) ✅

Notebook 2 executado (transformações definidas) ✅

Notebook 3 executado (stream iniciado + consultas aparecendo) ✅